<a href="https://colab.research.google.com/github/vidaechea/lidr-Master-ai-engineering/blob/development/session01/session_01_3_api_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google import genai
from google.genai import types
from google.genai.errors import APIError, ClientError

from google.colab import userdata

# It's recommended to load your API key from Colab secrets.
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=GEMINI_API_KEY)

# Gemini 2.5 Flash pricing (check periodically)
PRICING = {
    "gemini-2.5-flash": {"input": 0.15, "output": 0.60, "thinking": 3.50},
}

def query_gemini(
    message,
    system_instruction=None,
    model="gemini-2.5-flash",
    temperature=0.3
):
    """
    Make a call to the Gemini API and return the response
    along with relevant metadata.
    """
    try:
        config_params = {
            "temperature": temperature,
            "max_output_tokens": 1000,
        }
        if system_instruction:
            config_params["system_instruction"] = system_instruction

        response = client.models.generate_content(
            model=model,
            contents=message,
            config=types.GenerateContentConfig(**config_params)
        )

        # Check that the response was not blocked
        finish_reason = response.candidates[0].finish_reason.name
        if finish_reason == "SAFETY":
            return {"error": "Response blocked by safety filters"}

        # Calculate cost
        usage = response.usage_metadata
        prices = PRICING.get(model, {"input": 0, "output": 0, "thinking": 0})
        cost = (
            (usage.prompt_token_count / 1_000_000) * prices["input"] +
            (usage.candidates_token_count / 1_000_000) * prices["output"] +
            (usage.thoughts_token_count / 1_000_000) * prices["thinking"]
        )

        return {
            "content": response.text,
            "model": response.model_version,
            "input_tokens": usage.prompt_token_count,
            "output_tokens": usage.candidates_token_count,
            "thinking_tokens": usage.thoughts_token_count,
            "finish_reason": finish_reason,
            "cost_usd": cost,
        }

    except ClientError as e:
        return {"error": f"Bad request: {e}"}
    except APIError as e:
        if e.code == 429:
            return {"error": "Rate limit reached or insufficient quota"}
        return {"error": f"API error ({e.code}): {e.message}"}

# Usage
result = query_gemini(
    message="How long would a PostgreSQL to Aurora migration take?",
    system_instruction="You are a software estimation consultant. Be concise."
)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    print(result["content"])
    print(f"\n--- Metadata ---")
    print(f"Model: {result['model']}")
    print(f"Tokens: {result['input_tokens']} input + {result['output_tokens']} output + {result['thinking_tokens']} thinking")
    print(f"Finish reason: {result['finish_reason']}")
    print(f"Cost: ${result['cost_usd']:.6f}")

This varies significantly. Key factors are:

*   **Database size and complexity** (schema, data types, stored procedures, triggers)
*   **Application downtime tolerance** (dict

--- Metadata ---
Model: gemini-2.5-flash
Tokens: 22 input + 38 output + 958 thinking
Finish reason: MAX_TOKENS
Cost: $0.003379
